# Task4 — MixUp short fine-tuning pilot

Run All โหลด fold 1 ที่เสร็จแล้ว และ **ฝึก fold 2 เพิ่ม 4 epochs แล้วหยุด** ไม่มี folds3–5/full refit/Test/submission ในเล่มนี้. ใช้ matched EfficientNet seed42 checkpoint ของแต่ละ fold เท่านั้น; ห้ามใช้ full model เริ่ม OOF

MixUp probability .5, Beta(.2,.2), weighted soft-target cross entropy, LR2e-5, batch32, seed42, compound augmentation .6 เดิม. ภาพคู่มาจาก training batch เท่านั้น. อีกครึ่งของ batch ใช้ hard labels ตามปกติ

เปรียบเทียบ baseline 0.81047, MixUp เดี่ยว และ baseline75%+MixUp25% โดยใช้ validation rows/corruptions/TTA เดิม. คะแนน Public ไม่สามารถอนุมานตรง ๆ จาก pilot ได้

ประมาณฝึก 14 นาทีจากความเร็ว EfficientNet เดิม แต่ยังไม่รวม validation/stress. checkpoint ทุก epoch; รันซ้ำโหลดที่เสร็จแล้ว. หยุด ConvNeXt ก่อนเพื่อไม่แย่ง GPU

นี่คือการทดลอง regularization; ไม่มี no-MixUp continuation control ดังนั้นกำไรที่ได้ยังแยกจากผลของการฝึกเพิ่มไม่ได้. OOF/suite ถูกใช้ซ้ำ ผลเป็น exploratory

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_mixup_training as training
from Task4.models import image_mixup_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/mixup_pilot_v1'
BATCH_SIZE=32
NUM_WORKERS=0
NAME='mixup_efficientnet4'
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')
display(asdict(training.RECIPES[NAME]))

Device: mps


{'architecture': 'efficientnet_b0',
 'epochs': 4,
 'backbone_lr': 2e-05,
 'head_lr': 2e-05,
 'weight_decay': 0.0001,
 'schedule': 'cosine',
 'warmup_epochs': 1,
 'augmentation': 'mixup_compound',
 'label_smoothing': 0.0,
 'seed': 42,
 'balance': 'weighted_loss',
 'jsd_weight': 0.0,
 'compound_probability': 0.6,
 'mixup_probability': 0.5,
 'mixup_alpha': 0.2}

## 1. ตรวจ Train และ matched-fold checkpoints

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':asdict(training.RECIPES[NAME]),'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. Fine-tune fold 1 — 4 epochs
checkpoint ใช้ชื่อ run ใหม่ ไม่ทับงานเดิม

In [3]:
parent=evaluation.references(ROOT,1)[2]
started=time.monotonic()
frame=training.run_fold(RAW,groups,folds,NAME,1,OUT,BATCH_SIZE,NUM_WORKERS,init_path=parent)
print('Train + original OOF minutes:',round((time.monotonic()-started)/60,1))

fold1 epoch 1 / 4 loss 0.37559 seconds 216.1
fold1 epoch 2 / 4 loss 0.29369 seconds 205.8
fold1 epoch 3 / 4 loss 0.31742 seconds 211.3
fold1 epoch 4 / 4 loss 0.326 seconds 207.0
Train + original OOF minutes: 15.2


## 3. Paired validation และผลต่าง
Stress inference อาจนานกว่าการฝึก; reference caches โหลดกลับมาใช้เมื่อ provenance ตรงกัน

In [4]:
scores,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,1,frame)
display(evaluation.summarize(scores))
wide=scores.pivot(index='condition',columns='procedure',values='macro_F1')
display(wide.assign(mixup_gain=wide.mixup-wide.baseline,blend_gain=wide.mixup_blend25-wide.baseline))
display(overlap[overlap.condition.eq('original')])
# Family-balanced stress; don't let six compound cases dominate the mean.
s=scores.copy()
s['family']=s.condition.map(lambda c:'original' if c=='original' else ('compound' if c.startswith('compound_') else CONDITIONS[c][1]))
f=s.groupby(['procedure','family']).macro_F1.mean().unstack()
reports={}
for name in ['mixup','mixup_blend25']:
    delta=f.loc[name]-f.loc['baseline']
    reports[name]={'original_gain':float(delta.original),
                   'stress_family_gain':float(delta.drop('original').mean()),
                   'worth_reviewing':bool(delta.original>=-.001 and delta.drop('original').mean()>=.002)}
validation.write_json(OUT/'pilot_report.json',{'fold':1,'reports':reports,
    'automatic_continuation':False,'test_used':False,
    'note':'Single fold exploratory screening only; no claim of 5-fold improvement'})
display(reports)
print('Fold 1 complete/cached. Continue to section 4 for the authorized fold 2 experiment.')

stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fold1 jpeg_2
stress fold1 dark_1
stress fold1 dark_2
stress fold1 compound_dark_noise
stress fold1 compound_color_noise
stress fold1 compound_rotation_dark
stress fold1 compound_bright_color
stress fold1 compound_dark_color_noise
stress fold1 compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.944725,0.986331,0.977201
mixup,0.937834,0.979420,0.970447
mixup_blend25,0.947435,0.986807,0.978992


procedure,baseline,mixup,mixup_blend25,mixup_gain,blend_gain
condition,,,,,
blur_1,0.982007,0.974058,0.979976,-0.007949,-0.002031
blur_2,0.976005,0.966113,0.977972,-0.009891,0.001967
compound_bright_color,0.947935,0.944041,0.953959,-0.003894,0.006024
compound_color_noise,0.949996,0.946123,0.952164,-0.003872,0.002168
compound_dark_color_noise,0.927962,0.920313,0.938288,-0.007649,0.010326
compound_dark_noise,0.954018,0.944036,0.955954,-0.009983,0.001935
compound_rotation_color_noise,0.926192,0.920478,0.922125,-0.005714,-0.004067
compound_rotation_dark,0.962249,0.952015,0.962121,-0.010234,-0.000128
dark_1,0.982041,0.974006,0.982041,-0.008035,0.000000


,fold,phase,condition,rows,baseline_errors,candidate_errors,both_wrong,candidate_fixes,candidate_breaks
0,1,development,original,6310,70,109,57,13,52


{'mixup': {'original_gain': -0.006911659625904898,
  'stress_family_gain': -0.0067770662409601945,
  'worth_reviewing': False},
 'mixup_blend25': {'original_gain': 0.0004759019704139167,
  'stress_family_gain': 0.0019434958068727342,
  'worth_reviewing': False}}

STOP: pilot complete. No further training or Test inference in this notebook.


## 4. Fold 2 — confirmation ก่อนลงทุน folds ที่เหลือ

ใช้ recipe และน้ำหนัก ensemble เดิมทั้งหมด ไม่ปรับตามผล fold 1. เริ่มจาก EfficientNet **fold 2** ที่ไม่เคยฝึกบน validation ของ fold 2; เพิ่ม 4 epochs พร้อม checkpoint. ถ้า kernel เดิมยังอยู่ รันตั้งแต่เซลล์นี้ได้; ถ้า restart ให้ Run All เพื่อโหลด fold 1 กลับมา

เป็นการสำรวจบน validation ที่ใช้ซ้ำ ไม่ใช่หลักฐานรับรองคะแนน Kaggle. **หยุดหลัง fold 2** ไม่ฝึก full model

In [5]:
parent2=evaluation.references(ROOT,2)[2]
started=time.monotonic()
frame2=training.run_fold(RAW,groups,folds,NAME,2,OUT,BATCH_SIZE,NUM_WORKERS,init_path=parent2)
print('Fold 2 train + original OOF minutes:',round((time.monotonic()-started)/60,1))
scores2,overlap2=evaluation.compare(ROOT,RAW,groups,folds,OUT,2,frame2)
display(evaluation.summarize(scores2))

fold2 epoch 1 / 4 loss 0.384 seconds 208.9
fold2 epoch 2 / 4 loss 0.30023 seconds 207.4
fold2 epoch 3 / 4 loss 0.31783 seconds 206.8
fold2 epoch 4 / 4 loss 0.32618 seconds 209.2
Fold 2 train + original OOF minutes: 15.0
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress fold2 speckle_2
stress fold2 blur_1
stress fold2 blur_2
stress fold2 jpeg_1
stress fold2 jpeg_2
stress fold2 dark_1
stress fold2 dark_2
stress fold2 compound_dark_noise
stress fold2 compound_color_noise
stress fold2 compound_rotation_dark
stress fold2 compound_bright_color
stress fold2 compound_dark_color_noise
stress fold2 compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.955820,0.983375,0.985600
mixup,0.952755,0.975619,0.983362
mixup_blend25,0.958791,0.983676,0.984589


## 5. เทียบสอง folds และบันทึกผล
รายงานผลแต่ละ fold แยกจากค่าเฉลี่ย เพื่อไม่ให้ fold ที่ดีบดบัง fold ที่แย่. ยังไม่เรียกเกณฑ์ selection สำหรับ 5 folds และไม่ใช้ audit

In [6]:
parts=[pd.read_parquet(OUT/f'development_fold{k}_scores.parquet') for k in (1,2)]
combined=pd.concat(parts,ignore_index=True)
assert set(combined.fold)=={1,2} and combined.phase.eq('development').all()
assert not combined.duplicated(['fold','condition','procedure']).any()
combined.to_parquet(OUT/'two_fold_scores.parquet',index=False)
display(evaluation.summarize(combined))
s=combined.copy()
s['family']=s.condition.map(lambda c:'original' if c=='original' else ('compound' if c.startswith('compound_') else CONDITIONS[c][1]))
f=s.groupby(['procedure','fold','family']).macro_F1.mean()
rows=[]
for name in ['mixup','mixup_blend25']:
    delta=(f.loc[name]-f.loc['baseline']).unstack('family')
    for fold in (1,2):
        rows.append({'procedure':name,'fold':fold,'original_gain':float(delta.loc[fold,'original']),
                     'stress_family_gain':float(delta.drop(columns='original').loc[fold].mean())})
gains=pd.DataFrame(rows)
display(gains)
gains.to_parquet(OUT/'two_fold_gains.parquet',index=False)
condition_scores=combined.pivot(index=['fold','condition'],columns='procedure',values='macro_F1')
display(condition_scores.assign(blend_gain=condition_scores.mixup_blend25-condition_scores.baseline))
validation.write_json(OUT/'two_fold_report.json',{
    'folds':[1,2],'gains':rows,'test_used':False,'automatic_continuation':False,
    'note':'Exploratory two-fold comparison; not 5-fold confirmation or Public-score prediction'})
print('STOP: two folds complete. Folds 3–5, full refit and Test have not been started.')

suite,compound,original,single
procedure,,,
baseline,0.950273,0.984853,0.981401
mixup,0.945295,0.977520,0.976904
mixup_blend25,0.953113,0.985242,0.981790


,procedure,fold,original_gain,stress_family_gain
0,mixup,1,-0.006912,-0.006777
1,mixup,2,-0.007756,-0.002376
2,mixup_blend25,1,0.000476,0.001943
3,mixup_blend25,2,0.000301,-0.000348


procedure                           baseline     mixup  mixup_blend25  \
fold condition                                                          
1    blur_1                         0.982007  0.974058       0.979976   
     blur_2                         0.976005  0.966113       0.977972   
     compound_bright_color          0.947935  0.944041       0.953959   
     compound_color_noise           0.949996  0.946123       0.952164   
     compound_dark_color_noise      0.927962  0.920313       0.938288   
     compound_dark_noise            0.954018  0.944036       0.955954   
     compound_rotation_color_noise  0.926192  0.920478       0.922125   
     compound_rotation_dark         0.962249  0.952015       0.962121   
     dark_1                         0.982041  0.974006       0.982041   
     dark_2                         0.976084  0.968026       0.980023   
     gaussian_1                     0.977994  0.968062       0.979984   
     gaussian_2                     0.969964  0.970137       0.978008   
     jpeg_1                         0.973924  0.969937       0.977907   
     jpeg_2                         0.977957  0.967996       0.973966   
     original                       0.986331  0.979420       0.986807   
     speckle_1                      0.980017  0.974029       0.980024   
     speckle_2                      0.976023  0.972109       0.980017   
2    blur_1                         0.987989  0.987936       0.985968   
     blur_2                         0.982014  0.983965       0.983985   
     compound_bright_color          0.960388  0.956270       0.960306   
     compound_color_noise           0.964076  0.969996       0.972060   
     compound_dark_color_noise      0.942069  0.936082       0.948026   
     compound_dark_noise            0.950065  0.941862       0.950002   
     compound_rotation_color_noise  0.942321  0.936327       0.946361   
     compound_rotation_dark         0.976000  0.975994       0.975990   
     dark_1                         0.985968  0.983965       0.985978   
     dark_2                         0.984027  0.983975       0.981985   
     gaussian_1                     0.986009  0.981966       0.986009   
     gaussian_2                     0.982009  0.977943       0.980008   
     jpeg_1                         0.989990  0.981956       0.985959   
     jpeg_2                         0.985999  0.988009       0.987999   
     original                       0.983375  0.975619       0.983676   
     speckle_1                      0.992000  0.983965       0.986009   
     speckle_2                      0.979998  0.979935       0.981988   

procedure                           blend_gain  
fold condition                                  
1    blur_1                          -0.002031  
     blur_2                           0.001967  
     compound_bright_color            0.006024  
     compound_color_noise             0.002168  
     compound_dark_color_noise        0.010326  
     compound_dark_noise              0.001935  
     compound_rotation_color_noise   -0.004067  
     compound_rotation_dark          -0.000128  
     dark_1                           0.000000  
     dark_2                           0.003939  
     gaussian_1                       0.001990  
     gaussian_2                       0.008044  
     jpeg_1                           0.003983  
     jpeg_2                          -0.003990  
     original                         0.000476  
     speckle_1                        0.000007  
     speckle_2                        0.003994  
2    blur_1                          -0.002022  
     blur_2                           0.001971  
     compound_bright_color           -0.000082  
     compound_color_noise             0.007984  
     compound_dark_color_noise        0.005956  
     compound_dark_noise             -0.000063  
     compound_rotation_color_noise    0.004040  
     compound_rotation_dark          -0.000010  
     dark_1                           0.000010  

STOP: two folds complete. Folds 3–5, full refit and Test have not been started.
